In [ ]:
# Load necessary library
library(data.table)

# Define file paths
main_file <- "~/pids-drg-claims/data-cleaning/debug/main.csv"
refactor_file <- "~/pids-drg-claims/data-cleaning/debug/refactor.csv"
output_file <- "~/pids-drg-claims/data-cleaning/debug/merged_output.csv"

# Read CSV files as data.tables
main_dt <- fread(main_file, colClasses = "character")
refactor_dt <- fread(refactor_file, colClasses = "character")

# Remove the unwanted columns from main_dt
cols_to_remove <- c("is_covid", "c1", "c2", "clin_rvs")
main_dt <- main_dt[, !colnames(main_dt) %in% cols_to_remove, with = FALSE]

# Define join keys
keys <- c("id_year", "id_series", "id_pin", "id_hci", "id_hcp")
setkeyv(main_dt, keys)
setkeyv(refactor_dt, keys)

# Perform a full join
joined_dt <- merge(main_dt, refactor_dt, all = TRUE)

# Identify column names
all_cols <- colnames(joined_dt)
key_cols <- keys # Preserve join keys in original order

# Identify columns with ".x" and ".y" suffixes
x_cols <- grep("\\.x$", all_cols, value = TRUE) # Get all ".x" columns
y_cols <- gsub("\\.x$", ".y", x_cols) # Get matching ".y" columns

# Ensure that we only reorder if corresponding ".y" columns exist
y_cols <- y_cols[y_cols %in% all_cols]

# Create the new column order
ordered_cols <- c(
  key_cols, # Start with keys
  as.vector(rbind(x_cols, y_cols)) # Interleave .x and .y columns
)

# Reorder the data.table
setcolorder(joined_dt, ordered_cols)

# Create a logical vector to check where at least one `.x` and `.y` value differs
diff_rows <- joined_dt[, Reduce(`|`, Map(`!=`, .SD[, x_cols, with = FALSE], .SD[, y_cols, with = FALSE]))]

# Filter dataset to retain only differing rows
diff_dt <- joined_dt[diff_rows]

# Replace identical values in corresponding `.x` and `.y` cells with `NA_character_`
for (i in seq_along(x_cols)) {
  x_col <- x_cols[i]
  y_col <- y_cols[i]

  identical_mask <- diff_dt[[x_col]] == diff_dt[[y_col]]

  # Replace identical values with `NA_character_` (completely blank in CSV)
  diff_dt[[x_col]][identical_mask] <- NA_character_
  diff_dt[[y_col]][identical_mask] <- NA_character_
}

# Remove column pairs where both `.main` and `.refactor` columns are entirely empty
empty_cols <- sapply(seq_along(x_cols), function(i) {
  x_col <- x_cols[i]
  y_col <- y_cols[i]
  all(is.na(diff_dt[[x_col]]) & is.na(diff_dt[[y_col]]))
})

# Drop completely empty `.x` and `.y` column pairs
cols_to_keep <- setdiff(colnames(diff_dt), c(x_cols[empty_cols], y_cols[empty_cols]))
diff_dt <- diff_dt[, ..cols_to_keep]

# Rename remaining `.x` to `.main` and `.y` to `.refactor`
setnames(diff_dt, old = x_cols, new = gsub("\\.x$", ".main", x_cols), skip_absent = TRUE)
setnames(diff_dt, old = y_cols, new = gsub("\\.y$", ".refactor", y_cols), skip_absent = TRUE)

# Save result, ensuring NA values remain blank
fwrite(diff_dt, output_file, na = "")

# Print confirmation
print("Processing complete. Differences saved to output file.")
